In [ ]:
!pip install ctgan

In [ ]:
import pandas as pd
from ctgan import CTGAN
import numpy as np
random_state = 42


In [ ]:
df1 = pd.read_csv("https://raw.githubusercontent.com/senihberkay/US-Army-ANSUR-II/master/ANSUR%20II%20FEMALE%20Public.csv", encoding='latin1')

In [ ]:
df2 = pd.read_csv("https://raw.githubusercontent.com/senihberkay/US-Army-ANSUR-II/master/ANSUR%20II%20MALE%20Public.csv", encoding='latin1')

In [ ]:
df1.head()

In [ ]:
df2.head()

In [ ]:
df2.rename(columns={'subjectid':'SubjectId'}, inplace=True)

In [ ]:
df1.columns == df2.columns

In [ ]:
print(df1.columns.tolist())

In [ ]:
df = pd.concat([df1, df2], ignore_index=True)
display(df.head())

In [ ]:
df.info()

In [ ]:
df.set_index('SubjectId', inplace=True)
display(df.head())

In [ ]:
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]
display(missing_values.sort_values(ascending=False))

In [ ]:
columns_to_drop = ['Date', 'Installation', 'Component', 'Branch', 'PrimaryMOS', 'SubjectsBirthLocation', 'SubjectNumericRace', 'Ethnicity', 'DODRace', 'Gender', 'Heightin', 'Weightlbs', 'WritingPreference']
df.drop(columns=columns_to_drop, inplace=True)
display(df.head())

In [ ]:
duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")

In [ ]:
df.describe()

In [ ]:
bins = [10, 19, 29, 39, 49, 59]
labels = ['10-19', '20-29', '30-39', '40-49', '50-59']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels, right=True, include_lowest=True)

display(df[['Age', 'Age_Group']].head())

In [ ]:
df.drop(columns = "Age", inplace = True)

## Correlation Analysis for the Full Dataset (`df`)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Create a copy of df to avoid modifying the original and to add the encoded Age_Group
df_corr = df.copy()

# Initialize LabelEncoder
le = LabelEncoder()

# Fit the LabelEncoder on the 'Age_Group' column before transforming
le.fit(df_corr['Age_Group'])

# Encode the 'Age_Group' column using the already fitted LabelEncoder 'le'
df_corr['Age_Group_encoded'] = le.transform(df_corr['Age_Group'])

# Drop the original categorical 'Age_Group' column as we now have the encoded version
df_corr = df_corr.drop(columns=['Age_Group'])

# Calculate correlations with 'Age_Group_encoded'
df_correlations = df_corr.corr()['Age_Group_encoded'].drop('Age_Group_encoded')

print("Correlations with Age_Group for the Full Dataset (df):")
display(df_correlations.sort_values(ascending=False).head(10))
display(df_correlations.sort_values(ascending=True).head(10))

## Feature Selection: Top 25% Correlated Features with 'Age_Group'

In [ ]:
import numpy as np

# Ensure df_corr is updated in case there were changes to df since its last creation
df_corr = df.copy()
df_corr['Age_Group_encoded'] = le.transform(df_corr['Age_Group'])

# Drop the original categorical 'Age_Group' column before calculating correlations
df_corr = df_corr.drop(columns=['Age_Group'])

# Calculate absolute correlations with 'Age_Group_encoded'
absolute_correlations = df_corr.corr()['Age_Group_encoded'].abs().drop('Age_Group_encoded')

# Determine the number of top features (25%)
num_features_to_keep = int(len(absolute_correlations) * 0.25)

# Get the names of the top features
top_features = absolute_correlations.nlargest(num_features_to_keep).index.tolist()

print(f"Original number of features: {len(df_corr.columns) - 1}")
print(f"Number of features to keep (top 25%): {num_features_to_keep}")
print("Top features selected:")
display(top_features)

# Filter the original df to keep only these top features and the 'Age_Group' column
df = df[top_features + ['Age_Group']]

print("\nDataFrame 'df' after filtering by top correlated features:")
display(df.head())

In [ ]:
from sklearn.model_selection import train_test_split

df_train, df_val = train_test_split(df, test_size=0.7, random_state=42, stratify=df['Age_Group'])

print(f"Shape of training set (df_train): {df_train.shape}")
print(f"Shape of validation set (df_val): {df_val.shape}")

### Separate features (X) and target (y)

In [ ]:
TARGET_COLUMN = 'Age_Group'

X_train = df_train.drop(columns=[TARGET_COLUMN])
y_train = df_train[TARGET_COLUMN]

X_val = df_val.drop(columns=[TARGET_COLUMN])
y_val = df_val[TARGET_COLUMN]

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_val shape: {y_val.shape}")

In [ ]:
ctgan = CTGAN(epochs=10)
ctgan.fit(df_train, discrete_columns = ["Age_Group"])

# Create synthetic data
synthetic_data = ctgan.sample(20000)

In [ ]:
synthetic_data.groupby("Age_Group")["Age_Group"].count()

In [ ]:
X_syn = synthetic_data.drop(columns=[TARGET_COLUMN])
y_syn = synthetic_data[TARGET_COLUMN]

In [26]:
import numpy as np
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import rbf_kernel


print("--- THÔNG SỐ ĐẦU VÀO HỆ THỐNG ---")
print(f" -> Kích thước tập dữ liệu THẬT (X_train_np): {X_train.shape}")
print(f" -> Kích thước tập dữ liệu GIẢ   (X_synth):    {X_syn.shape}")


# =====================================================================
# STEP 2: THUẬT TOÁN KIỂM ĐỊNH PHÂN PHỐI ĐÃ KHẮC PHỤC LỖI QUY MÔ (DATA SCALE)
# =====================================================================

def calculate_corrected_synos_metrics(X_train, X_syn):
    """
    Hàm tính toán khoảng cách phân phối đồng bộ thang đo.
    Sử dụng StandardScaler của dữ liệu THẬT để đưa cả hai tập về cùng phân phối (Mean=0, Std=1).
    """
    # 1. Tiến hành Z-score Scaling để triệt tiêu ảnh hưởng đơn vị đo thô (mm)
    scaler = StandardScaler()
    X_real_scaled = scaler.fit_transform(X_train)
    X_synth_scaled = scaler.transform(X_syn)

    # 2. Đo lường Wasserstein Distance đơn biến trên dữ liệu đã Chuẩn hóa
    num_features = X_real_scaled.shape[1]
    wd_scaled_list = []

    for i in range(num_features):
        wd = wasserstein_distance(X_real_scaled[:, i], X_synth_scaled[:, i])
        wd_scaled_list.append(wd)

    mean_wd_scaled = np.mean(wd_scaled_list)
    # Quy đổi khoảng cách chuẩn hóa sang tỉ lệ % độ tin cậy hình thái cột
    wd_percentage = np.exp(-mean_wd_scaled) * 100

    # 3. Đo lường MMD đa biến trên không gian phân phối gốc (RBF Kernel tự kiểm soát biên độ)
    gamma = 1.0 / X_train.shape[1]
    K_real_real = rbf_kernel(X_train, X_train, gamma=gamma)
    K_synth_synth = rbf_kernel(X_syn, X_syn, gamma=gamma)
    K_real_synth = rbf_kernel(X_train, X_syn, gamma=gamma)

    mmd_squared = K_real_real.mean() + K_synth_synth.mean() - 2 * K_real_synth.mean()
    raw_mmd = np.sqrt(max(0, mmd_squared))
    # Quy đổi khoảng cách đa biến sang tỉ lệ % độ tin cậy tương quan giải phẫu
    mmd_percentage = np.exp(-raw_mmd) * 100

    # 4. Tính điểm số tin cậy phân phối tổng hợp (Overall Score)
    overall_trust_score = (wd_percentage + mmd_percentage) / 2

    return mean_wd_scaled, wd_percentage, raw_mmd, mmd_percentage, overall_trust_score


# =====================================================================
# STEP 3: THỰC THI CHẤM ĐIỂM & ĐÓNG GÓI KẾT QUẢ BÁO CÁO NGHIÊN CỨU
# =====================================================================

# Chạy thuật toán xử lý đồng bộ
raw_wd, wd_pct, raw_mmd, mmd_pct, overall_score = calculate_corrected_synos_metrics(X_train, X_syn)

# Xuất bảng hiển thị kết quả chuẩn mực khoa học
print("\n" + "="*55)
print("   KẾT QUẢ KIỂM ĐỊNH PHÂN PHỐI DỮ LIỆU TỔNG HỢP HOÀN CHỈNH")
print("="*55)
print(f"1. Wasserstein Distance Metric (Đơn biến - Đã chuẩn hóa):")
print(f"   - Khoảng cách thô định lượng: {raw_wd:.4f}")
print(f"   - Độ tin cậy hình thái cột   : {wd_pct:.2f}%")
print(f"     (Tránh lỗi tràn số mũ của dữ liệu trắc lượng thô)\n")
print(f"2. Maximum Mean Discrepancy Metric (Đa biến toàn cục):")
print(f"   - Khoảng cách thô tương quan: {raw_mmd:.4f}")
print(f"   - Độ tin cậy tương quan giải phẫu (Allometry): {mmd_pct:.2f}%\n")
print("-" * 55)
print(f"--> OVERALL DISTRIBUTION TRUST SCORE: {overall_score:.2f}%")
print("="*55)

--- THÔNG SỐ ĐẦU VÀO HỆ THỐNG ---
 -> Kích thước tập dữ liệu THẬT (X_train_np): (1820, 23)
 -> Kích thước tập dữ liệu GIẢ   (X_synth):    (20000, 23)

   KẾT QUẢ KIỂM ĐỊNH PHÂN PHỐI DỮ LIỆU TỔNG HỢP HOÀN CHỈNH
1. Wasserstein Distance Metric (Đơn biến - Đã chuẩn hóa):
   - Khoảng cách thô định lượng: 0.5023
   - Độ tin cậy hình thái cột   : 60.52%
     (Tránh lỗi tràn số mũ của dữ liệu trắc lượng thô)

2. Maximum Mean Discrepancy Metric (Đa biến toàn cục):
   - Khoảng cách thô tương quan: 0.0245
   - Độ tin cậy tương quan giải phẫu (Allometry): 97.58%

-------------------------------------------------------
--> OVERALL DISTRIBUTION TRUST SCORE: 79.05%


In [ ]:
!pip install sdmetrics

In [ ]:
from sdmetrics.reports.single_table import QualityReport

In [29]:
num_features = df_train.shape[1]




# =====================================================================
# STEP 2: KHỞI TẠO METADATA THEO TIÊU CHUẨN SDMETRICS
# =====================================================================
# SDMetrics cần biết cột nào là số liên tục (numerical) và cột nào là phân loại (categorical)
metadata = {
    'columns': {col: {'sdtype': 'numerical'} for col in df_train.columns if col != 'Age_Group'},
}
# Bổ sung thông tin cột gender là biến phân loại (nhãn encoded số nguyên)
metadata['columns']['Age_Group'] = {'sdtype': 'categorical'}


# =====================================================================
# STEP 3: CHẠY BÁO CÁO CHẤT LƯỢNG VÀ TRÍCH XUẤT SỐ % ĐỘ TIN CẬY
# =====================================================================
print("--- ĐANG TIẾN HÀNH CHẤM ĐIỂM VỚI SDMETRICS ---")

# Khởi tạo và tạo báo cáo
report = QualityReport()
report.generate(df_train, synthetic_data, metadata)

# 1. Lấy điểm số phần trăm tổng quát (Overall Score)
overall_score_pct = report.get_score() * 100

# 2. Lấy điểm số chi tiết theo từng khía cạnh cốt lõi
properties_df = report.get_properties()

print("\n" + "="*55)
print("   BÁO CÁO CHẤT LƯỢNG DỮ LIỆU TỔNG HỢP TỪ SDMETRICS")
print("="*55)
for index, row in properties_df.iterrows():
    property_name = row['Property']
    score_pct = row['Score'] * 100

    # Dịch nghĩa các thuộc tính của SDMetrics sang ngôn ngữ luận án SYNOS
    if property_name == 'Column Shapes':
        print(f"1. Độ tương đồng Hình dáng Đơn biến (Column Shapes): {score_pct:.2f}%")
        print("   (Bao gồm kiểm định KS-Test cho biến số và Chi-Square cho giới tính)")
    elif property_name == 'Column Pair Trends':
        print(f"\n2. Độ tương đồng Tương quan Đa biến (Column Pair Trends): {score_pct:.2f}%")
        print("   (Kiểm tra ma trận tương quan chéo giữ luật giải phẫu Allometry)")

print("-" * 55)
print(f"--> OVERALL SDMETRICS TRUST SCORE: {overall_score_pct:.2f}%")
print("="*55)

--- ĐANG TIẾN HÀNH CHẤM ĐIỂM VỚI SDMETRICS ---
Generating report ...

(1/2) Evaluating Column Shapes: |██████████| 24/24 [00:00<00:00, 135.12it/s]|
Column Shapes Score: 80.0%

(2/2) Evaluating Column Pair Trends: |█         | 30/276 [00:00<00:05, 42.33it/s]|

/usr/local/lib/python3.12/dist-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:103: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_real_counts = real.groupby(list(columns), dropna=False).size()


(2/2) Evaluating Column Pair Trends: |█▊        | 49/276 [00:01<00:04, 50.87it/s]|

/usr/local/lib/python3.12/dist-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:103: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_real_counts = real.groupby(list(columns), dropna=False).size()


(2/2) Evaluating Column Pair Trends: |██▌       | 70/276 [00:01<00:05, 37.04it/s]|

...

/usr/local/lib/python3.12/dist-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:103: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_real_counts = real.groupby(list(columns), dropna=False).size()


(2/2) Evaluating Column Pair Trends: |█████████ | 251/276 [00:05<00:00, 60.34it/s]|

/usr/local/lib/python3.12/dist-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:103: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_real_counts = real.groupby(list(columns), dropna=False).size()
/usr/local/lib/python3.12/dist-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:103: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_real_counts = real.groupby(list(columns), dropna=False).size()
/usr/local/lib/python3.12/dist-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:103: FutureWarning: The default of observed=False is deprecat

(2/2) Evaluating Column Pair Trends: |██████████| 276/276 [00:06<00:00, 44.86it/s]|
Column Pair Trends Score: 64.37%

Overall Score (Average): 72.19%


   BÁO CÁO CHẤT LƯỢNG DỮ LIỆU TỔNG HỢP TỪ SDMETRICS
1. Độ tương đồng Hình dáng Đơn biến (Column Shapes): 80.00%
   (Bao gồm kiểm định KS-Test cho biến số và Chi-Square cho giới tính)

2. Độ tương đồng Tương quan Đa biến (Column Pair Trends): 64.37%
   (Kiểm tra ma trận tương quan chéo giữ luật giải phẫu Allometry)
-------------------------------------------------------
--> OVERALL SDMETRICS TRUST SCORE: 72.19%


/usr/local/lib/python3.12/dist-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:103: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_real_counts = real.groupby(list(columns), dropna=False).size()
/usr/local/lib/python3.12/dist-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:103: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  contingency_real_counts = real.groupby(list(columns), dropna=False).size()
/usr/local/lib/python3.12/dist-packages/sdmetrics/column_pairs/statistical/contingency_similarity.py:103: FutureWarning: The default of observed=False is deprecat

In [30]:

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# =====================================================================
# DATA PREPROCESSING & DATASET SYNCHRONISATION
# =====================================================================
target_col = 'Age_Group'
label_mapping = {'10-19': 0, '20-29': 1,'30-39': 2, '40-49': 3,'50-59': 4}

def extract_features_labels(df, target, mapping=None):
    df_proc = df.copy()
    if mapping:
        df_proc[target] = df_proc[target].map(mapping)
    return df_proc.drop(columns=[target]).values, df_proc[target].values.astype(int)

# Extract real and synthetic experimental matrices
X_train_real, y_train_real = extract_features_labels(df_train, target_col, label_mapping)
X_val_real, y_val_real = extract_features_labels(df_val, target_col, label_mapping)
X_train_synth, y_train_synth = extract_features_labels(synthetic_data, target_col, label_mapping)

# =====================================================================
# MODEL ARCHITECTURE CONFIGURATION
# =====================================================================
models_config = {
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=5, random_state=42, eval_metric='logloss'),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42)
}

# =====================================================================
# EVALUATION EXECUTION (Baseline TRTR vs. Generative TSTR)
# =====================================================================
report_data = []

def evaluate_pipeline(model, X_train, y_train, X_val):
    model.fit(X_train, y_train)
    preds = model.predict(X_val)

    # Elegant extraction of probability or decision scores for AUROC calculation
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(X_val) # Return full probability array for multi-class
    else:
        probs = getattr(model, "decision_function", lambda x: None)(X_val) # Return None if no decision_function either
    return preds, probs

for name, model in models_config.items():
    # 1. Baseline Configuration (Train Real, Test Real)
    y_pred_real, y_prob_real = evaluate_pipeline(model, X_train_real, y_train_real, X_val_real)

    # 2. Synthetic Configuration (Train Synthetic, Test Real)
    y_pred_synth, y_prob_synth = evaluate_pipeline(model, X_train_synth, y_train_synth, X_val_real)

    report_data.append({
        "Model": name,
        "Acc_Real": accuracy_score(y_val_real, y_pred_real),
        "F1_Real": f1_score(y_val_real, y_pred_real, average='macro'),
        "AUC_Real": roc_auc_score(y_val_real, y_prob_real, multi_class='ovr', average='macro'), # Added multi_class and average
        "Acc_Synth": accuracy_score(y_val_real, y_pred_synth),
        "F1_Synth": f1_score(y_val_real, y_pred_synth, average='macro'),
        "AUC_Synth": roc_auc_score(y_val_real, y_prob_synth, multi_class='ovr', average='macro') # Added multi_class and average
    })

# =====================================================================
# PERFORMANCE COMPILATION & GENERATIVE EFFICACY ANALYSIS
# =====================================================================
df_res = pd.DataFrame(report_data).set_index("Model")

for metric in ['Acc', 'F1', 'AUC']:
    df_res[f'{metric}_Efficacy (%)'] = (df_res[f'{metric}_Synth'] / df_res[f'{metric}_Real']) * 100

print("\n" + "="*85)
print("             BẢNG TỔNG HỢP HIỆU NĂNG ĐỘ TIN CẬY (TSTR vs TRTR)")
print("="*85)

format_dict = {col: '{:,.2f}%'.format for col in df_res.columns if 'Efficacy' in col}
other_cols = {col: '{:,.4f}'.format for col in df_res.columns if 'Efficacy' not in col}

print(df_res.style.format({**format_dict, **other_cols}).to_string())
print("="*85)


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c


             BẢNG TỔNG HỢP HIỆU NĂNG ĐỘ TIN CẬY (TSTR vs TRTR)
 Acc_Real F1_Real AUC_Real Acc_Synth F1_Synth AUC_Synth Acc_Efficacy (%) F1_Efficacy (%) AUC_Efficacy (%)
Model         
XGBoost 0.5311 0.2701 0.6997 0.4136 0.1726 0.4630 77.88% 63.89% 66.17%
Random Forest 0.5341 0.1851 0.7152 0.5210 0.1370 0.4754 97.53% 74.03% 66.46%
Logistic Regression 0.5523 0.2759 0.7225 0.5214 0.1371 0.5047 94.42% 49.69% 69.85%
SVM 0.5214 0.1371 0.7337 0.5214 0.1371 0.4293 100.00% 100.00% 58.50%



In [ ]:
df_res

In [ ]:
# Kiểm tra xem dữ liệu giả có bị trùng với dữ liệu thật không
# Sử dụng real_df thay vì df_train vì real_df có cùng cấu trúc cột và kiểu dữ liệu (đặc biệt là 'Gender') với synth_df
duplicates = pd.merge(synthetic_data, df, how='inner')
print(f"Số lượng bản ghi bị sao chép y hệt: {len(duplicates)}")

In [ ]:
# 1. Kiểm tra sự trùng lặp giữa Train và Val
intersection = pd.merge(df_train, df_val, how='inner')
print(f"Số lượng bản ghi trùng lặp giữa Train và Val: {len(intersection)}")

# 2. Kiểm tra các cột có tính chất định danh
# Tìm các cột có số lượng giá trị duy nhất cao bằng số lượng mẫu
for col in df_train.columns:
    if df_train[col].nunique() == len(df_train):
        print(f"Cảnh báo: Cột '{col}' có thể là biến ID gây rò rỉ dữ liệu!")